# ☕ LBLM368 — Verisi doğrulanan kahve çekirdeği CNN deneyi

**Bu sürüm, Kaggle arşivinde `train/test` varsa validation ayırma hatasını da düzeltir.** Sırayla hücreleri çalıştır. T4 GPU önerilir. Gerçek eğitim çalışmadan rapor tamamlanmış sayılmaz. GitHub güncelleme isteğe bağlıdır.


## 1. Kaynak dosyalarını çıkar

In [ ]:
from pathlib import Path
import base64, io, zipfile
PROJECT=Path('/content/ruken_kahve_proje')
PROJECT.mkdir(parents=True,exist_ok=True)
EMBEDDED = base64.b64decode(
    '...'
)
with zipfile.ZipFile(io.BytesIO(EMBEDDED)) as z:
    z.extractall(PROJECT)
print('✅ Dosyalar:', sorted(p.name for p in PROJECT.iterdir()))


## 2. Gerekli paketleri kur

In [ ]:
import sys, subprocess, shutil
subprocess.run([sys.executable,'-m','pip','-q','install','-r',str(PROJECT/'requirements.txt')],check=True)
if not (shutil.which('libreoffice') or shutil.which('soffice')):
    subprocess.run(['apt-get','-qq','update'],check=False)
    subprocess.run(['apt-get','-qq','-y','install','libreoffice-writer'],check=False)
print('✅ Bağımlılıklar yüklendi. PDF dönüştürücü:',shutil.which('soffice'))


## 3. Veriyi indir, 10 epoch eğit, ölç, doğrula ve raporu üret

In [ ]:
# Bu hücre tüm gerçek eğitimi, test doğrulamasını ve rapor üretimini yapar.
# Olası hatanın TAMAMINI yazdırır: yalnızca CalledProcessError görünmez.
import subprocess, sys
result=subprocess.run([sys.executable, str(PROJECT/'run_all.py')],
                     cwd=PROJECT,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout[-20000:])
if result.returncode != 0:
    raise RuntimeError('Eğitim/rapor aşaması başarısız. Hata açıklaması yukarıda. Çalıştırma kodu: '+str(result.returncode))
print('✅ Gerçek eğitim ve rapor dosyaları üretildi.')


## 4. Sonuçları incele ve ZIP indir

In [ ]:
# Yalnızca 3. hücre BAŞARILI OLURSA bu adım çalışır.
import json
from google.colab import files
from IPython.display import display,Image
results=json.loads((PROJECT/'artifacts'/'sonuclar.json').read_text(encoding='utf-8'))
print('GERÇEK TEST METRİKLERİ:',json.dumps(results['metrics'],indent=2,ensure_ascii=False))
print('EĞİTİM/DOĞRULAMA/TEST:',results['dataset_counts'])
display(Image(filename=str(PROJECT/'artifacts'/'ornek_tahminler.png'),width=760))
files.download(str(PROJECT/'LBLM368_Ruken_YILDIZ_GERCEK_DENEY_TESLIM_PAKETI.zip'))


## 5. (İsteğe bağlı) GitHub deposunu güncelle

In [ ]:
# İSTEĞE BAĞLI: GITHUB_TOKEN gizli anahtarı olmadan repo değişmez.
# Bu hücreyi ancak deneyi başarıyla tamamladıktan sonra kullan.
import os,subprocess,sys
try:
    from google.colab import userdata
    token=userdata.get('GITHUB_TOKEN')
except Exception:
    token=None
if token:
    env=os.environ.copy();env['GITHUB_TOKEN']=token
    result=subprocess.run([sys.executable,str(PROJECT/'github_guncelle.py')],cwd=PROJECT,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    print(result.stdout[-7000:])
    if result.returncode:raise RuntimeError('GitHub senkronizasyonu başarısız.')
    print('✅ GitHub güncellendi.')
else:
    print('🔒 GITHUB_TOKEN eklenmedi. GitHub değiştirilmedi. Önce ZIP dosyasını indirin.')
